In [ ]:
!pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---------------------------------------- 9.3/9.3 MB 58.3 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 47.3 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 7.2/7.2 MB 55.7 MB/s  0:00:00

   ---------------------------------------- 0/7 [pyparsing]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------------------- 1/7 [pillow]
   ----- ---------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\clara\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import math
import matplotlib.pyplot as plt

In [ ]:
# Mathematical modeling of the problem

# 1. Objective Function (tank cost)
def f(r, h):
    return 350 * math.pi * (r**2) + 200 * math.pi * r * h

# 2. Inequality Constraint (minimum volume)
def g(r, h):
    return 500 - math.pi * (r**2) * h

# 3. Penalty Function
def P(r, h, mu=1e6):
    # the max() function returns 0 if the constraint is respected (g <= 0),
    # or returns the value of g if the constraint is violated (g > 0).
    penalidade = max(0, g(r, h))
    
    # Adds the cost to the squared penalty
    return f(r, h) + mu * (penalidade**2)

In [ ]:
# 4. Numerical Partial Derivatives (Forward Finite Differences)
def calculate_partial_derivatives(r: float, h: float, mu: float = 1e6, delta: float = 1e-5) -> tuple:
    """
    Approximates the partial derivatives of the penalty function P(r, h) 
    with respect to the radius (r) and height (h) using forward finite differences.
    
    Args:
        r (float): Current tank radius.
        h (float): Current tank height.
        mu (float): Penalty coefficient for the volume constraint.
        delta (float): Small perturbation value for the finite difference approximation.
        
    Returns:
        tuple: A tuple containing the partial derivatives (dP_dr, dP_dh).
    """
    dP_dr = (P(r + delta, h, mu) - P(r, h, mu)) / delta
    dP_dh = (P(r, h + delta, mu) - P(r, h, mu)) / delta
    
    return dP_dr, dP_dh

In [ ]:
# 5. Gradient Descent Optimization Algorithm
def gradient_descent_optimizer(r_init: float, h_init: float, alpha: float = 1e-5, 
                               mu: float = 1e6, delta: float = 1e-5, 
                               max_iter: int = 50000, tol: float = 1e-6) -> tuple:
    """
    Minimizes the tank cost function using the Gradient Descent method, 
    subject to the volume constraint managed by a penalty function.
    
    Args:
        r_init (float): Initial guess for the radius.
        h_init (float): Initial guess for the height.
        alpha (float): Learning rate (step size).
        mu (float): Penalty coefficient for constraint violations.
        delta (float): Perturbation step for finite differences.
        max_iter (int): Maximum number of iterations allowed.
        tol (float): Tolerance for the stopping criterion.
        
    Returns:
        tuple: Optimal radius, optimal height, and tracking history lists for (r, h, cost).
    """
    r, h = r_init, h_init
    
    # Tracking history for convergence analysis and plotting
    history_r, history_h, history_cost = [r], [h], [f(r, h)]
    
    for k in range(max_iter):
        dP_dr, dP_dh = calculate_partial_derivatives(r, h, mu, delta)
        
        # Update design variables by moving against the gradient direction
        r_new = r - alpha * dP_dr
        h_new = h - alpha * dP_dh
        
        # Physical boundary constraint: dimensions must remain strictly positive
        r_new = max(0.1, r_new)
        h_new = max(0.1, h_new)
        
        # Stopping criterion: convergence check based on parameter variation
        if abs(r_new - r) < tol and abs(h_new - h) < tol:
            print(f"Convergence achieved! Optimization completed at iteration {k}.")
            break
            
        r, h = r_new, h_new
        
        # Append current state to history logs
        history_r.append(r)
        history_h.append(h)
        history_cost.append(f(r, h))
        
    return r, h, history_r, history_h, history_cost

In [ ]:
# 6. Execution and Model Evaluation
# Define initial guesses for the design variables
initial_radius = 5.0
initial_height = 10.0

# Execute the optimization algorithm
r_opt, h_opt, hist_r, hist_h, hist_cost = gradient_descent_optimizer(
    r_init=initial_radius, 
    h_init=initial_height, 
    alpha=1e-5,   # Learning rate (step size)
    mu=1e6        # Penalty constant
)

# Compute final metrics based on the optimized parameters
final_volume = math.pi * (r_opt**2) * h_opt
final_cost = f(r_opt, h_opt)

# Display formatted results
print("-" * 50)
print("COMPUTATIONAL OPTIMIZATION RESULTS")
print("-" * 50)
print(f"Optimal Radius (r) : {r_opt:.4f} m")
print(f"Optimal Height (h) : {h_opt:.4f} m")
print(f"Final Tank Volume  : {final_volume:.2f} m³ (Minimum Required: 500.00 m³)")
print(f"Minimum Total Cost : R$ {final_cost:.2f}")
print("-" * 50)